# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset (Croissant schema) with the `mlcroissant` library.

### Dataset Source
This dataset describes protein abundance, modifications, and sequences in human samples, acquired by mass spectrometry on extracellular vesicles from stimulated mast cells. Metadata and record structure follow the [Croissant specification](https://mlcommons.github.io/croissant/).

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Viewing basic metadata
print("Dataset loaded!")
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("License:", dataset.metadata.license)
print("Published date:", dataset.metadata.datePublished)
print("Record sets in schema:", dataset.metadata.recordSet)

if hasattr(dataset.metadata, "keywords"):
    print("Keywords:", ', '.join(dataset.metadata.keywords))

## 2. Data Overview
We'll list available record sets, fields, and columns by their `@id` (unique identifier). This overview helps select the entities for later extraction and processing.

In [ ]:
# Show all available record sets and their @id
if hasattr(dataset.metadata, "recordSet") and dataset.metadata.recordSet:
    print("Available record sets:")
    recordset_objs = dataset.metadata.recordSet
    # Handle both list or single object
    if not isinstance(recordset_objs, list):
        recordset_objs = [recordset_objs]
    for rs in recordset_objs:
        if hasattr(rs, '@id'):
            print(f"- Record set @id: {rs['@id']}")
        else:
            print(f"- Record set: {rs}")

    # For the first record set, print its fields and columns
    # We'll assume the main set is at index 0
    main_rs = recordset_objs[0]
    # Try to extract fields from the record set
    if 'field' in main_rs:
        print('\nFields in the main record set:')
        fields = main_rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for fld in fields:
            if isinstance(fld, dict) and '@id' in fld:
                print(f"- Field @id: {fld['@id']} name: {fld.get('name','')} (dataType: {fld.get('dataType','')})")
            else:
                print(f"- Field: {fld}")
else:
    print('No record sets defined directly in metadata. Attempting to discover records by enumerating...')
    # Use the dataset utility to list record set ids
    try:
        # mlcroissant's Dataset._record_set_ids is intended for exploration
        all_rs_ids = dataset._record_set_ids()
        print('Record set @ids from the schema:')
        for r in all_rs_ids:
            print('-', r)

        # Let us peek at the field names of one record set
        example_rs = all_rs_ids[0]
        print(f'\nFields in record set: {example_rs}')
        records = list(dataset.records(record_set=example_rs))
        if records:
            print('Sample record keys:', list(records[0].keys()))
        else:
            print('No records found in record set', example_rs)
    except Exception as e:
        print('Could not enumerate record sets:', e)

## 3. Data Extraction
We'll extract table(s) from selected record set(s) into pandas DataFrames for analysis. All references use `@id` values for record set and fields as required. 

In [ ]:
# Discover all available record set `@id`s
record_set_ids = dataset._record_set_ids()
print('Discovered record sets:', record_set_ids)

# For demonstration, pick the main record set containing proteins (likely the first one)
main_record_set_id = record_set_ids[0]
print(f"\nUsing main record set: {main_record_set_id}")

# Load records from ALL record sets for completeness
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records from {rsid} ...")
    records = list(dataset.records(record_set=rsid))
    if len(records) == 0:
        print(f"No records found in {rsid}")
    else:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Show the columns of the main record set and example records
if main_record_set_id in dataframes:
    print(f"\nFields/columns in {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("Main record set not found in loaded dataframes!")

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group the data using their field `@id`s. For the main proteins record set, we'll process a numeric field such as 'mw' (molecular weight), assuming its field `@id` is present, and group by 'description' if available.

All processing refers only to fields from the data, using the DataFrame columns (which correspond to the Croissant schema field IDs).

In [ ]:
# 1. Pick a numeric field to filter and normalize
main_df = dataframes[main_record_set_id]
numeric_candidates = [c for c in main_df.columns if main_df[c].dtype in ['float64', 'int64']]
print('Numeric field options:', numeric_candidates)

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")
    threshold = main_df[numeric_field].mean() # or any suitable value, e.g. 10000
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.1f} (mean): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (added as {numeric_field}_normalized):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # 2. Group by another field (categorical/text)
    group_candidates = [c for c in main_df.columns if main_df[c].dtype == 'object' and c != numeric_field]
    print('Group-by field candidates:', group_candidates)
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}, top 5:")
        display(grouped_df.head())
else:
    print('No numeric fields found in main record set; no EDA performed.')

## 5. Visualization

Let's plot the distribution of the main numeric field, and a boxplot by group if available. Visualization will use the field `@id`s in plot titles for traceability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if group_field exists
    if 'group_field' in locals():
        top_groups = filtered_df[group_field].value_counts().index[:5]
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_groups)])
        plt.title(f"{numeric_field} by {group_field} (top 5)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field to visualize.')

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the Croissant FAIR² dataset for proteins from human mast cell vesicles via the mlcroissant library.

- The metadata, record sets, and fields are directly referenced by their Croissant `@id`.
- We've loaded one or more record sets into DataFrames, filtered and normalized a numeric field, and visualized its distribution.
- You can repeat these steps for any field/column of interest by using its `@id` as seen in section 2.

For further exploration, consult the Croissant schema documentation or the [mlcroissant library reference](https://pypi.org/project/mlcroissant/) for more processing and interoperability options.